# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
import itertools
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv("../ex01/day-of-week-not-scaled.csv")
df2 = pd.read_csv("../ex00/dayofweek.csv")
df['dayofweek'] = df2['dayofweek']
print(df.shape)
print(df.dtypes.to_list())
df.head()

(1686, 44)
[dtype('int64'), dtype('int64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('int64')]


,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4


In [3]:
random_state=21
test_size=0.2
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=random_state, test_size=test_size, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, random_state=random_state, test_size=test_size, stratify=y_train)

print('X_train shape: ', X_train.shape)
print('y_train shape: ', y_train.shape)
print('X_valid shape: ', X_valid.shape)
print('y_valid shape: ', y_valid.shape)
print('X_test shape : ', X_test.shape)
print('y_test shape : ', y_test.shape)

X_train shape:  (1078, 43)
y_train shape:  (1078,)
X_valid shape:  (270, 43)
y_valid shape:  (270,)
X_test shape :  (338, 43)
y_test shape :  (338,)


## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [4]:
model1 = SVC(random_state=random_state, probability=True, C=10, class_weight=None, gamma='auto', kernel='rbf')
model1.fit(X_train, y_train)
predict = model1.predict(X_valid)

accuracy = accuracy_score(y_valid, predict)
precision = precision_score(y_valid, predict, average='weighted')
recall = recall_score(y_valid, predict, average='weighted')

print(f"accuracy: {accuracy:.5f}")
print(f"precision: {precision:.5f}")
print(f"recall: {recall:.5f}")

accuracy: 0.87778
precision: 0.88162
recall: 0.87778


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [5]:
model1 = SVC(random_state=random_state, probability=True, C=10, class_weight=None, gamma='auto', kernel='rbf')
model2 = DecisionTreeClassifier(random_state=random_state, class_weight='balanced', criterion='gini', max_depth=21)
model3 = RandomForestClassifier(random_state=random_state, class_weight='balanced', criterion='entropy', max_depth=24, n_estimators=100)

In [6]:
voting_clf = VotingClassifier(
    estimators=[
        ('svc', model1),
        ('tree', model2),
        ('rf', model3)
    ],
    voting='soft'
)

voting_clf.fit(X_train, y_train)
y_val_pred = voting_clf.predict(X_valid)

accuracy = accuracy_score(y_val_pred, y_valid)
precision = precision_score(y_val_pred, y_valid, average='weighted')
recall = recall_score(y_val_pred, y_valid, average='weighted')

print(f"Accuracy: {accuracy:.5f}")
print(f"Precision: {precision:.5f}")
print(f"Recall: {recall:.5f}")

Accuracy: 0.88519
Precision: 0.88586
Recall: 0.88519


In [7]:
weights_list = list(itertools.product([1, 2, 3, 4], repeat=3))

results = []

for weights in weights_list:
    voting_clf = VotingClassifier(
        estimators=[
            ('svc', model1),
            ('tree', model2),
            ('rf', model3)
        ],
        voting='soft',
        weights=weights
    )
    voting_clf.fit(X_train, y_train)
    y_val_pred = voting_clf.predict(X_valid)

    accuracy = accuracy_score(y_val_pred, y_valid)
    precision = precision_score(y_val_pred, y_valid, average='weighted')
    recall = recall_score(y_val_pred, y_valid, average='weighted')
    
    results.append({
        'weights': weights,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall
    })

results_sorted = sorted(results, key=lambda x: (x['accuracy'], x['precision']), reverse=True)

In [8]:
best_weights = results_sorted[0]['weights']
print(f"Best weights: {best_weights}")

Best weights: (4, 1, 4)


In [9]:
final_voting_clf = VotingClassifier(
    estimators=[
        ('svc', model1),
        ('tree', model2),
        ('rf', model3)
    ],
    voting='soft',
    weights=best_weights
)

final_voting_clf.fit(X_train, y_train)
y_test_pred = final_voting_clf.predict(X_test)

accuracy = accuracy_score(y_test_pred, y_test)
precision = precision_score(y_test_pred, y_test, average='weighted')
recall = recall_score(y_test_pred, y_test, average='weighted')

print(f"Test Accuracy: {accuracy:.5f}")
print(f"Test Precision: {precision:.5f}")
print(f"Test Recall: {recall:.5f}")

Test Accuracy: 0.90533
Test Precision: 0.90979
Test Recall: 0.90533


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [10]:
base_svm = SVC(random_state=random_state, probability=True, C=10, class_weight=None, gamma='auto', kernel='rbf')

bagging = BaggingClassifier(
    base_estimator=base_svm,
    n_estimators=10,
    random_state=21
)

bagging.fit(X_train, y_train)

y_pred = bagging.predict(X_valid)

accuracy = accuracy_score(y_valid, y_pred)
precision = precision_score(y_valid, y_pred, average='weighted')
recall = recall_score(y_valid, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.5f}")
print(f"Precision: {precision:.5f}")
print(f"Recall: {recall:.5f}")

Accuracy: 0.88519
Precision: 0.89427
Recall: 0.88519


In [11]:
param_grid = {
    'n_estimators' : [50],
    'max_samples': [0.5, 0.7, 1.0],
    'max_features': [0.5, 0.7, 1.0],
    'bootstrap': [True, False],
    'bootstrap_features': [True, False]
}

grid_bagging = GridSearchCV(
    estimator=BaggingClassifier(base_estimator=base_svm, random_state=21),
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

grid_bagging.fit(X_train, y_train)

print("Best parameters:", grid_bagging.best_params_)
print("Best accuracy:", grid_bagging.best_score_)

Best parameters: {'bootstrap': False, 'bootstrap_features': False, 'max_features': 1.0, 'max_samples': 1.0, 'n_estimators': 50}
Best accuracy: 0.8422695951765717


In [12]:
best_bagging = grid_bagging.best_estimator_

y_pred_test = best_bagging.predict(X_test)

accuracy = accuracy_score(y_test, y_pred_test)
precision = precision_score(y_test, y_pred_test, average='weighted')
recall = recall_score(y_test, y_pred_test, average='weighted')

print(f"Test Accuracy: {accuracy:.5f}")
print(f"Test Precision: {precision:.5f}")
print(f"Test Recall: {recall:.5f}")

Test Accuracy: 0.88462
Test Precision: 0.88658
Test Recall: 0.88462


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [13]:
estimators = [
    ('svc', model1),
    ('tree', model2),
    ('rf', model3)
]

results = []

for n_splits in [2, 3, 4, 5, 6, 7]:
    for passthrough in [True, False]:
        
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)
        
        stacking_clf = StackingClassifier(
            estimators=estimators,
            final_estimator=LogisticRegression(solver='liblinear'),
            cv=cv,
            passthrough=passthrough,
            n_jobs=-1
        )
        
        stacking_clf.fit(X_train, y_train)
        y_pred_val = stacking_clf.predict(X_valid)
        
        acc = accuracy_score(y_valid, y_pred_val)
        prec = precision_score(y_valid, y_pred_val, average='weighted')
        rec = recall_score(y_valid, y_pred_val, average='weighted')
        
        results.append({
            'n_splits': n_splits,
            'passthrough': passthrough,
            'accuracy': acc,
            'precision': prec,
            'recall': rec
        })
        
        print(f"n_splits={n_splits}, passthrough={passthrough} -> acc: {acc:.5f}, prec: {prec:.5f}, rec: {rec:.5f}")

n_splits=2, passthrough=True -> acc: 0.90370, prec: 0.90619, rec: 0.90370
n_splits=2, passthrough=False -> acc: 0.89630, prec: 0.89678, rec: 0.89630
n_splits=3, passthrough=True -> acc: 0.90370, prec: 0.90632, rec: 0.90370
n_splits=3, passthrough=False -> acc: 0.89630, prec: 0.89759, rec: 0.89630
n_splits=4, passthrough=True -> acc: 0.91111, prec: 0.91327, rec: 0.91111
n_splits=4, passthrough=False -> acc: 0.90370, prec: 0.90570, rec: 0.90370
n_splits=5, passthrough=True -> acc: 0.90000, prec: 0.90217, rec: 0.90000
n_splits=5, passthrough=False -> acc: 0.90000, prec: 0.90056, rec: 0.90000
n_splits=6, passthrough=True -> acc: 0.90370, prec: 0.90450, rec: 0.90370
n_splits=6, passthrough=False -> acc: 0.90370, prec: 0.90436, rec: 0.90370
n_splits=7, passthrough=True -> acc: 0.90370, prec: 0.90640, rec: 0.90370
n_splits=7, passthrough=False -> acc: 0.90370, prec: 0.90538, rec: 0.90370


In [14]:
df_results = pd.DataFrame(results)
best_model_params = df_results.sort_values(by=['accuracy', 'precision'], ascending=False).iloc[0]
print(best_model_params)

n_splits              4
passthrough        True
accuracy       0.911111
precision      0.913269
recall         0.911111
Name: 4, dtype: object


In [15]:
best_n_splits = int(best_model_params['n_splits'])
best_passthrough = bool(best_model_params['passthrough'])

cv_best = StratifiedKFold(n_splits=best_n_splits, shuffle=True, random_state=21)

stacking_best = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(solver='liblinear'),
    cv=cv_best,
    passthrough=best_passthrough,
    n_jobs=-1
)

stacking_best.fit(X_train, y_train)
y_pred_test = stacking_best.predict(X_test)

test_acc = accuracy_score(y_test, y_pred_test)
test_prec = precision_score(y_test, y_pred_test, average='weighted')
test_rec = recall_score(y_test, y_pred_test, average='weighted')

print(f"Test Accuracy: {test_acc:.5f}")
print(f"Test Precision: {test_prec:.5f}")
print(f"Test Recall: {test_rec:.5f}")

Test Accuracy: 0.90533
Test Precision: 0.90844
Test Recall: 0.90533


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [ ]:
best_model = final_voting_clf
y_test_pred = final_voting_clf.predict(X_test)

accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred, average='weighted')
recall = recall_score(y_test, y_test_pred, average='weighted')

print(f"Test Accuracy: {accuracy:.5f}")
print(f"Test Precision: {precision:.5f}")
print(f"Test Recall: {recall:.5f}")

Test Accuracy: 0.90533
Test Precision: 0.90881
Test Recall: 0.90533


In [ ]:
X_test_copy = X_test.copy()

X_test_copy['true'] = y_test
X_test_copy['pred'] = y_test_pred
X_test_copy['error'] = X_test_copy['true'] != X_test_copy['pred']

user_columns = [col for col in X_test.columns if col.startswith('uid_user_')]
X_test_copy['user'] = X_test_copy[user_columns].idxmax(axis=1).str.replace('uid_user_', '').astype(int)

lab_columns = [col for col in X_test.columns if col.startswith('labname_')]
X_test_copy['labname'] = X_test_copy[lab_columns].idxmax(axis=1).str.replace('labname_', '')



In [33]:
if 'weekday' in X_test_copy.columns:
    weekday_errors = X_test_copy.groupby('weekday')['error'].agg(['sum', 'count'])
    weekday_errors['error_rate'] = weekday_errors['sum'] / weekday_errors['count'] * 100
    weekday_errors_sorted = weekday_errors.sort_values(by='error_rate', ascending=False)
    print("Errors by weekday:")
    print(weekday_errors_sorted)

labname_errors = X_test_copy.groupby('labname')['error'].agg(['sum', 'count'])
labname_errors['error_rate'] = labname_errors['sum'] / labname_errors['count'] * 100
labname_errors_sorted = labname_errors.sort_values(by='error_rate', ascending=False)
print("\nErrors by labname:")
labname_errors_sorted


Errors by labname:


,sum,count,error_rate
labname,,,
lab03,1,1,100.000000
laba04,9,35,25.714286
laba04s,6,25,24.000000
lab05s,1,6,16.666667
laba06,1,9,11.111111
code_rvw,1,13,7.692308
laba06s,1,15,6.666667
project1,12,186,6.451613
lab03s,0,1,0.000000


In [31]:
user_errors = X_test_copy.groupby('user')['error'].agg(['sum', 'count'])
user_errors['error_rate'] = user_errors['sum'] / user_errors['count'] * 100
user_errors_sorted = user_errors.sort_values(by='error_rate', ascending=False)
print("\nErrors by user:")
user_errors_sorted.head(20)


Errors by user:


,sum,count,error_rate
user,,,
6,2,4,50.000000
17,2,7,28.571429
3,3,14,21.428571
16,1,5,20.000000
18,1,6,16.666667
27,1,6,16.666667
19,3,19,15.789474
2,4,28,14.285714
30,1,8,12.500000


In [34]:
joblib.dump(best_model, 'best_voting_model.joblib')

['best_voting_model.joblib']